# Olist Featurization Workflow

This notebook executes the current featurization pipeline for `olist_migration`.
The pipeline produces the final model-ready SP 2017 week-product matrix that is consumed by the clustering workflow.

The implementation lives in `featurization_scripts/featurization.py`, and the pipeline is driven by `featurizer_config.yaml`.


In [ ]:
import sys
import yaml
from pathlib import Path

import pandas as pd
from featurization.notebook_utils import build_notebook_resolver
from featurization.feature_advisor_util import FeatureAdvisorUtil, FeatureAdvisorPromptConfig

workspace_root = Path.cwd()
if not (workspace_root / 'featurizer_config.yaml').exists():
    if (workspace_root / 'notebooks' / 'featurizer_config.yaml').exists():
        workspace_root = workspace_root / 'notebooks'
    elif (workspace_root.parent / 'featurizer_config.yaml').exists():
        workspace_root = workspace_root.parent

for lib_dir in ['python3.13', 'python3.12', 'python3.11', 'python3.10']:
    pkg_dir = workspace_root / '.venv' / 'lib' / lib_dir / 'site-packages'
    if pkg_dir.exists():
        sys.path.insert(0, str(pkg_dir))
        break

config_path = workspace_root / 'featurizer_config.yaml'
with config_path.open('r', encoding='utf-8') as fp:
    config = yaml.safe_load(fp)

print('Workspace root:', workspace_root)
print('featurizer_config.yaml exists:', config_path.exists())
print('Merged raw source exists:', (workspace_root / 'data' / 'raw_olist_example_dataset.csv').exists())
print('Cleaned fallback source exists:', (workspace_root / 'data' / 'dd_cleaner' / 'olist_daily_orders_prepared_clean.csv').exists())
print('Loaded featurization config keys:', list(config.keys()))
print('Pipeline stages:')
for stage in config.get('pipeline', []):
    print('-', stage.get('name'), '->', stage.get('method'))

notebook_dir = workspace_root
if notebook_dir.name == 'notebooks' and (notebook_dir.parent / 'featurizer_config.yaml').exists():
    notebook_dir = notebook_dir
elif (notebook_dir / 'notebooks').is_dir() and (notebook_dir / 'featurizer_config.yaml').exists():
    notebook_dir = notebook_dir / 'notebooks'

resolver = build_notebook_resolver(str(notebook_dir))
print('Resolved notebook workspace root:', resolver.working_dir)
print('Dataset structural type:', resolver.structural_type)
print('Dataset structural type (raw config):', config.get('structural_type'))

metadata_path = Path(resolver.metadata_path)
input_data_path = Path(resolver.featurization_input_path)
print('Metadata path:', metadata_path)
print('Input data path:', input_data_path)

if not metadata_path.exists():
    raise FileNotFoundError(
        f'Metadata file not found at {metadata_path}. Update `metadata_file` in featurizer_config.yaml or place the file there.'
    )
if not input_data_path.exists():
    raise FileNotFoundError(
        f'Input data file not found at {input_data_path}. Run the featurization pipeline or update `featurization_input_data` in featurizer_config.yaml.'
    )

metadata = pd.read_csv(metadata_path)
input_data = pd.read_csv(input_data_path)
print('Metadata rows:', len(metadata))
print('Input dataset shape:', input_data.shape)
if resolver.structural_type == 'wide and short' or input_data.shape[1] > input_data.shape[0]:
    print('Dataset guidance: this appears to be a wide and short dataset.')
    print('Prefer lower-dimensional encodings and train-safe featurizations that reduce overfitting rather than high-dimensional sparse encodings.')

prompt_config = FeatureAdvisorPromptConfig.load_from_package()
advisor = FeatureAdvisorUtil(resolver=resolver, prompt_config=prompt_config)

print('Advisor output directory:', advisor.feature_advisor_dir)
print('Recommendation CSV path:', advisor.recommendations_csv_path)
print('Recommendation MD path:', advisor.recommendations_md_path)

recommendations = advisor.recommend(
    metadata=metadata,
    model_intent='graph',
    input_data=input_data,
    use_rules=True,
)

print('Recommendations generated by rule-based advisor')
print(recommendations.head(10).to_string(index=False))


Workspace root: /home/rajiv/programming/kmds_migration/olist_migration
featurizer_config.yaml exists: True
Merged raw source exists: True
Cleaned fallback source exists: True
Loaded featurization config keys: ['working_dir', 'structural_type', 'country_code', 'featurization_input_data', 'metadata_file', 'featurization_output_dir', 'sp_freq_prod_file', 'sp_freq_prod_parquet', 'pipeline']
Pipeline stages:
- Load Featurization Input Dataset -> load_raw_data
- Drop incomplete order rows -> validate_required_columns
- Build SP 2017 Weekly Product Matrix -> build_sp_weekly_product_matrix


In [2]:
import sys
from pathlib import Path

workspace_root = Path.cwd()
if not (workspace_root / "featurizer_config.yaml").exists():
    workspace_root = workspace_root.parent

sys.path.insert(0, str(workspace_root / "featurization_scripts"))

from featurization import run_pipeline

pipeline_result = run_pipeline(workspace_root / "featurizer_config.yaml")
print("Pipeline completed.")
print("Resolved output directory:", pipeline_result.coord.output_dir)
print("Weekly product matrix CSV:", pipeline_result.coord.sp_freq_prod_path)
print("Weekly product matrix parquet:", pipeline_result.coord.sp_freq_prod_parquet)


Running stage: Load Featurization Input Dataset
Loading featurization input dataset from /home/rajiv/programming/kmds_migration/olist_migration/data/dd_cleaner/olist_daily_orders_prepared_clean.csv
Running stage: Drop incomplete order rows
Running stage: Build SP 2017 Weekly Product Matrix
Pipeline completed.
Resolved output directory: /home/rajiv/programming/kmds_migration/olist_migration/data
Weekly product matrix CSV: /home/rajiv/programming/kmds_migration/olist_migration/data/SP_2017_freq_prod_weekly_sales_prepared.csv
Weekly product matrix parquet: /home/rajiv/programming/kmds_migration/olist_migration/data/SP_2017_freq_prod_weekly_sales_prepared.parquet


In [3]:
import pandas as pd
from pathlib import Path

paths = {
    "weekly_product_matrix_csv": pipeline_result.coord.sp_freq_prod_path,
    "weekly_product_matrix_parquet": pipeline_result.coord.sp_freq_prod_parquet,
}

for name, path in paths.items():
    print(f"\n{name}: {path}")
    if path.exists():
        if path.suffix == ".csv":
            df = pd.read_csv(path)
        else:
            df = pd.read_parquet(path)
        print(f"  rows={len(df)} cols={len(df.columns)}")
        print(df.head(3))
    else:
        print("  MISSING")


SyntaxError: unterminated f-string literal (detected at line 10) (2052027312.py, line 10)

## Notes

- The current pipeline only needs the final SP 2017 product-week matrix file.
- The featurization stage produces `data/SP_2017_freq_prod_weekly_sales_prepared.csv` and optionally `.parquet`.
- Legacy intermediate outputs are no longer required for modeling.
- The clustering workflow uses the final matrix file directly.
